# Packages

In [ ]:
import numpy as np
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen
from statsmodels.tsa.stattools import adfuller, grangercausalitytests, coint
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error
import statsmodels.api as sm

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
%load_ext nb_black

# Data

In [ ]:
# Load US macro economic data from statsmodels
data = sm.datasets.macrodata.load_pandas().data
data

In [ ]:
# Build a PeriodIndex from year and quarter
date_period = pd.PeriodIndex(
    year=data["year"].astype(int),
    quarter=data["quarter"].astype(int),
    freq="Q"
)

In [ ]:
# Convert PeriodIndex to timestamp (end of quarter)
data["date"] = date_period.to_timestamp()

In [ ]:
data

# EDA

In [ ]:
# Train / test split
data_train = data.loc[data['date'] < '2001-01-01']
data_test = data.loc[data['date'] >= '2001-01-01']

In [ ]:
# Set dataset name for combination later
data_train.loc[:, 'dataset'] = 'train'
data_test.loc[:, 'dataset'] = 'test'

In [ ]:
# Combine data for easier plotting later
data_combined = pd.concat([data_train, data_test], ignore_index=True)
data_combined['date'] = pd.to_datetime(data_combined['date'])

In [ ]:
# Some basic EDA
print(data_train.dtypes)
print(data_train.isnull().sum())
print(data_train.describe())

In [ ]:
# Plot our data through time
variables = ['realgdp', 'realcons']

for var in variables:
    fig = px.line(
        data_combined,
        x='date',
        y=var,
        color='dataset',
        title=f"{var.capitalize()} Over Time",
        labels={'date': 'Date', var: var.capitalize()},
        color_discrete_map={'train': 'blue', 'test': 'orange'},
        template="simple_white"
    )
    fig.show()

# Vector Autoregression

## Stationarity Check

In [ ]:
def adf_test(series):
    test_results = adfuller(series)
    print('ADF Statistic: ', test_results[0])
    print('P-Value: ', test_results[1])
    print('Critical Values:')
    for thres, adf_stat in test_results[4].items():
        print('\t%s: %.2f' % (thres, adf_stat))


In [ ]:
# Check the stationarity of the first dataset
adf_test(data_train['realgdp'])

In [ ]:
# Check the stationarity of the second data
adf_test(data_train['realcons'])

## Make Data Stationary

In [ ]:
data_train.set_index('date', inplace=True)
data_test.set_index('date', inplace=True)

In [ ]:
data_train_log = np.log(data_train[['realgdp', 'realcons']])
data_test_log = np.log(data_test[['realgdp', 'realcons']])

In [ ]:
data_train_log_diff = data_train_log[['realgdp', 'realcons']].diff().dropna()
data_test_log_diff = data_test_log[['realgdp', 'realcons']].diff().dropna()

In [ ]:
adf_test(data_train_log_diff['realgdp'])

In [ ]:
adf_test(data_train_log_diff['realcons'])

In [ ]:
# Visually check if our new data is stationary
fig = px.line(
    data_train_log_diff,
    x=data_train_log_diff.index,
    y='realgdp',
    title='Log and Difference of realgdp Over Time',
    labels={'date': 'Date', 'realgdp': 'Real gross domestic product (log and diff)'},
    template="simple_white"
)
fig.show()

In [ ]:
# Visually check if our new data is stationary
fig = px.line(
    data_train_log_diff,
    x=data_train_log_diff.index,
    y='realcons',
    title='Log and Difference of realcons Over Time',
    labels={'date': 'Date', 'realcons': 'Real personal consumption (log and diff)'},
    template="simple_white"
)
fig.show()

## Select Lags & Fit Model

In [ ]:
data_train_log_diff

In [ ]:
model = VAR(data_train_log_diff)
lag_order = model.select_order(maxlags=20)

In [ ]:
lag_order.summary()

In [ ]:
best_lag = lag_order.bic
best_lag

In [ ]:
best_lag = lag_order.bic
model_var = model.fit(best_lag)

In [ ]:
model_var.summary()

## Granger Causality Test

In [ ]:
# Function to run Granger causality and store p-values
def granger_causality_summary(df, maxlag):
    results = []
    for lag in range(1, maxlag + 1):
        test_result = grangercausalitytests(df, maxlag=lag, verbose=False)
        f_pvalue = test_result[lag][0]['ssr_ftest'][1]
        results.append({'lag': lag, 'p_value': f_pvalue})
    return pd.DataFrame(results)

In [ ]:
# Does realcons Granger-cause realgdp?
df1 = granger_causality_summary(data_train_log_diff[['realgdp', 'realcons']], maxlag=best_lag)
df1.rename(columns={'p_value': 'realcons_causes_realgdp_p'}, inplace=True)
df1

In [ ]:
# Does realgdp Granger-cause realcons?
df2 = granger_causality_summary(data_train_log_diff[['realcons', 'realgdp']], maxlag=best_lag)
df2.rename(columns={'p_value': 'realgdp_causes_realcons_p'}, inplace=True)
df2

## Forecast

In [ ]:
# Carry out the forecast
forecast_input = data_train_log_diff.values[-best_lag:]
n_periods = len(data_test_log_diff)
fc_log_diff = model_var.forecast(y=forecast_input, steps=n_periods)

In [ ]:
# Forecast is in first log differences
fc_log_diff = pd.DataFrame(fc_log_diff, index=data_test_log_diff.index,
                           columns=['realgdp', 'realcons'])

In [ ]:
# Undo differencing: cumulative sum + last observed log value
last_train_log = data_train_log.iloc[-1] 
fc_log = fc_log_diff.cumsum() + last_train_log

In [ ]:
# Undo log transform to get original units
fc_final = np.exp(fc_log)
fc_final

## Analysis

In [ ]:
# Get the actual values
y_true_realgdp = data_test['realgdp'].iloc[1:]
y_true_realcons = data_test['realcons'].iloc[1:]

In [ ]:
# Get the predicted values
y_pred_realgdp = fc_final['realgdp']
y_pred_realcons = fc_final['realcons']

In [ ]:
dates = fc_final.index

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("realgdp Forecast", "realcons Forecast"),
    shared_xaxes=True
)

fig.add_trace(go.Scatter(
    x=dates,
    y=y_pred_realgdp,
    mode='lines',
    name='Forecast realgdp'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates,
    y=y_true_realgdp,
    mode='lines',
    name='Actual realgdp',
    line=dict(color='orange')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates,
    y=y_pred_realcons,
    mode='lines',
    name='Forecast realcons'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=dates,
    y=y_true_realcons,
    mode='lines',
    name='Actual realcons',
    line=dict(color='red')
), row=2, col=1)

fig.update_layout(
    height=700,
    width=900,
    template="simple_white",
    legend_title="Legend",
    xaxis_title="Date",
    yaxis_title="Value",
)

fig.show()

# VECM

## Model

In [ ]:
# Use the VAR in level to find the best number of lags
model_var = VAR(data_train_log[['realgdp', 'realcons']])
lag_order_results = model_var.select_order(maxlags=20)
lag_order_results.summary()

In [ ]:
best_lag = lag_order_results.bic
best_lag

## Engle-Granger Cointegration Test

In [ ]:
# Engle-granger two step method cointegration test
score, pvalue, crit_value = coint(data_train_log['realgdp'],
                                  data_train_log['realcons'])

In [ ]:
print("Test statistic:", score)
print("p-value:", pvalue)
print("Critical values:", crit_value)


## Johansen Test

In [ ]:
# Johansen cointegration test
johansen_test = coint_johansen(data_train_log[['realgdp', 'realcons']], det_order=1, k_ar_diff=20-1)

In [ ]:
# Trace statistic and 5% critical values
print('Trace statistic:', johansen_test.lr1)
print('Critical values (5%):', johansen_test.cvt[:, 1])

## Fit Model

In [ ]:
# Fit the VECM
vecm = VECM(endog=data_train_log[['realgdp', 'realcons']], k_ar_diff=1, coint_rank=1, deterministic='ct')
vecm_fit = vecm.fit()

In [ ]:
vecm_fit.summary()

## Forecast

In [ ]:
fc = vecm_fit.predict(steps=len(data_test_log))
fc_df = pd.DataFrame(fc, columns=data_train_log[['realgdp', 'realcons']].columns)

In [ ]:
fc_original = np.exp(fc_df)

In [ ]:
# Get the forecasts values
y_realgdp_vecm = fc_original['realgdp'].iloc[1:]
y_realcons_vecm = fc_original['realcons'].iloc[1:]

## Analysis

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("realgdp Forecast", "realcons Forecast"),
    shared_xaxes=True
)

fig.add_trace(go.Scatter(
    x=dates,
    y=y_pred_realgdp,
    mode='lines',
    name='Forecast realgdp VAR'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates,
    y=y_realgdp_vecm,
    mode='lines',
    name='Forecast realgdp VECM'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates,
    y=y_true_realgdp,
    mode='lines',
    name='Actual realgdp',
    line=dict(color='orange')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates,
    y=y_pred_realcons,
    mode='lines',
    name='Forecast realcons VAR'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=dates,
    y=y_realcons_vecm,
    mode='lines',
    name='Forecast realcons VECM'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=dates,
    y=y_true_realcons,
    mode='lines',
    name='Actual realcons',
    line=dict(color='red')
), row=2, col=1)

fig.update_layout(
    height=700,
    width=900,
    template="simple_white",
    legend_title="Legend",
    xaxis_title="Date",
    yaxis_title="Value",
)

fig.show()